# 15 · Export more demo inputs
One cell. Downloads a zip with ~34 real PTB-XL test-fold ECG files (.npy, plus a picture of each) and 30+ prescription images to upload into the website.

In [ ]:
from google.colab import drive, files
import numpy as np, pandas as pd, os, glob, shutil, sys
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/CardioSafe'; sys.path.insert(0, WORK)
OUT = '/content/demo_inputs'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT + '/ecg'); os.makedirs(OUT + '/ecg_pictures'); os.makedirs(OUT + '/prescriptions')
N_PER_CLASS, CLASSES = 6, ['NORM', 'MI', 'STTC', 'CD', 'HYP']
rng = np.random.default_rng(7)

# ---------- 1. ECG signals from the PTB-XL test fold (fold 10), never used for training
md = pd.read_csv(f'{WORK}/ptbxl_meta_super5.csv'); print('meta columns:', list(md.columns))
fold = next(c for c in md.columns if 'fold' in c.lower()); sexc = next((c for c in md.columns if c.lower() == 'sex'), None)
idc = next((c for c in md.columns if 'ecg_id' in c.lower()), None)
import re
supc = next(c for c in md.columns if 'super' in c.lower())
cc = {}
for c in CLASSES:
    md['_' + c] = md[supc].astype(str).apply(lambda v, c=c: int(c in re.findall(r'[A-Za-z]+', v))); cc[c] = '_' + c
print('label columns built from', supc, '| example:', md[supc].iloc[0], '| counts:', {c: int(md[cc[c]].sum()) for c in CLASSES})
SIG = None
for f in sorted(glob.glob(f'{WORK}/**/*.np[yz]', recursive=True)):
    if '/preds/' in f or '/models/' in f: continue
    try:
        z = np.load(f, allow_pickle=True, mmap_mode='r') if f.endswith('.npy') else np.load(f, allow_pickle=True)
        for k, a in ([('arr', z)] if f.endswith('.npy') else [(k, z[k]) for k in z.files]):
            if getattr(a, 'ndim', 0) == 3 and tuple(a.shape[1:]) == (1000, 12) and a.shape[0] in (len(md), int((md[fold] == 10).sum())):
                SIG = (f, k, a); break
    except Exception: pass
    if SIG: break
assert SIG, 'could not find the full 12-lead signal array in MyDrive/CardioSafe (looked for shape N x 1000 x 12)'
f, k, A = SIG; print('signals from', os.path.relpath(f, WORK), k, A.shape)
rows = np.where(md[fold].values == 10)[0] if A.shape[0] == len(md) else np.arange(A.shape[0])
sub = md.iloc[rows] if A.shape[0] == len(md) else md[md[fold] == 10]
Y = sub[[cc[c] for c in CLASSES]].values.astype(int)
pick = []
for j, c in enumerate(CLASSES):                       # records with ONLY this class (clean examples)
    cand = np.where((Y[:, j] == 1) & (Y.sum(1) == 1))[0]; pick += list(rng.choice(cand, min(N_PER_CLASS, len(cand)), replace=False))
multi = np.where(Y.sum(1) >= 2)[0]; pick += list(rng.choice(multi, 4, replace=False))   # a few multi-label cases
pick = list(dict.fromkeys(pick)); print(len(pick), 'ECG records selected')
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
index = []
for n, i in enumerate(pick, 1):
    x = np.asarray(A[rows[i]] if A.shape[0] == len(md) else A[i], dtype='float32')
    labs = '+'.join(c for j, c in enumerate(CLASSES) if Y[i, j]); sx = 'male' if sexc and int(sub.iloc[i][sexc]) == 0 else 'female'
    eid = int(sub.iloc[i][idc]) if idc else i
    base = f'ecg_{n:02d}_{labs}_{sx}_id{eid}'
    np.save(f'{OUT}/ecg/{base}.npy', x)
    fig, ax = plt.subplots(6, 2, figsize=(11, 8), sharex=True); t = np.arange(1000) / 100
    for kk, a in enumerate(ax.T.flatten()): a.plot(t, x[:, kk], lw=.7, color='#1f3a5f'); a.set_ylabel(LEADS[kk], rotation=0, labelpad=14, fontsize=8); a.set_yticks([])
    fig.suptitle(f'{base}  (dataset label: {labs}, sex: {sx})', fontsize=9); plt.tight_layout(); plt.savefig(f'{OUT}/ecg_pictures/{base}.png', dpi=110); plt.close()
    index.append({'file': base + '.npy', 'dataset_label': labs, 'sex': sx, 'ecg_id': eid})
pd.DataFrame(index).to_csv(f'{OUT}/ecg_index.csv', index=False)
print('signal range check (mV): mean |x| =', round(float(np.mean([np.abs(np.load(f)).mean() for f in glob.glob(OUT + "/ecg/*.npy")[:5]])), 3), '(should be well below 5; ~0.1-0.3 is typical)')

# ---------- 2. Synthetic printed prescriptions (same generator used for the OCR benchmark)
import cardiosafe_rx as R
rows_b = []
bm = f'{WORK}/kb/brand_map.csv'
if os.path.exists(bm):
    b = pd.read_csv(bm); rows_b = [(str(r[0]), str(r[1]), '') for r in b.itertuples(index=False)]
pool = R.build_pool(rows_b); print('prescription pool:', len(pool), 'drugs')
rx_index = []
for n in range(1, 31):
    level = 'easy' if n <= 20 else 'medium'
    lines, truth = R.make_prescription(pool, rng, decoys=1)
    img = R.render(lines, rng, level)
    name = f'rx_{n:02d}_{level}.jpg'; img.save(f'{OUT}/prescriptions/{name}', quality=92)
    rx_index.append({'file': name, 'level': level, 'drugs': '; '.join(t['name'] for t in truth)})
pd.DataFrame(rx_index).to_csv(f'{OUT}/prescription_index.csv', index=False)
for f in sorted(glob.glob(f'{WORK}/demo/rx_example_*.jpg')): shutil.copy(f, OUT + '/prescriptions')

shutil.make_archive('/content/demo_inputs', 'zip', OUT)
print('ECG files:', len(os.listdir(OUT + '/ecg')), '| prescriptions:', len(os.listdir(OUT + '/prescriptions')))
files.download('/content/demo_inputs.zip')
